# UD Treebank Cross-linguistic Typological Metadata

This notebook demonstrates the **UD Treebank Cross-linguistic Typological Metadata** dataset, which provides a single joinable metadata table covering Universal Dependencies treebanks.

Each row contains:
- **WALS Feature 81A** word-order classification (SOV/SVO/VSO/No dominant order)
- **Glottolog** language family, family ID, and macroarea
- **Morphological case richness** (distinct Case= values and token proportion)
- **Head-direction entropy** in binary and per-deprel weighted variants
- **Modality labels** (spoken/written/mixed)
- **Differential object marking** flags
- Basic stats (sentences, tokens, mean sentence length)

The original script loads raw treebank metadata and formats it into a schema-compliant structure for downstream analysis.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import sys
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Load Data

Load the mini demo dataset from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-e4150b-head-directionality-dependent-temporal-d/main/dataset_iter1_ud_treebank_cro/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {data['metadata']['total_treebanks']} treebanks")
print(f"Source: {data['metadata']['source']}")
print(f"External sources: {data['metadata']['external_sources']}")

## Configuration

Tunable parameters for the demo. `MAX_EXAMPLES` controls how many treebank records to process (set to `None` for all).

In [ ]:
# Maximum number of examples to process (None = all)
MAX_EXAMPLES = None  # Demo dataset is small (40 examples), process all

## Extract Examples

Extract examples from the loaded data and apply the processing limit.

In [ ]:
examples = data["datasets"][0]["examples"]
if MAX_EXAMPLES is not None:
    examples = examples[:MAX_EXAMPLES]

print(f"Processing {len(examples)} treebank examples")
print(f"\nFirst example (treebank ID = '{examples[0]['input']}'):")
print(f"  Language: {examples[0]['metadata_language_name']}")
print(f"  Word order: {examples[0]['metadata_wals_word_order'] or '(unknown)'}")
print(f"  Family: {examples[0]['metadata_glottolog_family_name']}")
print(f"  Macroarea: {examples[0]['metadata_macroarea']}")
print(f"  Modality: {examples[0]['metadata_modality']}")
print(f"  Case richness: {examples[0]['metadata_case_richness_count']}")
print(f"  Head-direction entropy: {examples[0]['metadata_head_direction_entropy_binary']}")

## Validation Summary

Compute coverage and distribution statistics, mirroring the original script's validation logic.

In [ ]:
# ── Validation summary (mirrors original data.py logic) ──────────────────────
n = len(examples)
modalities = {}
word_orders = {}
for e in examples:
    m = e["metadata_modality"]
    modalities[m] = modalities.get(m, 0) + 1
    wo = e["metadata_wals_word_order"]
    if wo:
        word_orders[wo] = word_orders.get(wo, 0) + 1

wals_coverage = sum(1 for e in examples if e["metadata_wals_word_order"])
glottolog_coverage = sum(1 for e in examples if e["metadata_glottolog_family_name"])

print(f"Total examples: {n}")
print(f"\nModalities: {modalities}")
print(f"Word orders: {word_orders}")
print(f"\nWALS coverage: {wals_coverage}/{n} ({100*wals_coverage/n:.1f}%)")
print(f"Glottolog coverage: {glottolog_coverage}/{n} ({100*glottolog_coverage/n:.1f}%)")

## Build DataFrame

Convert the examples to a pandas DataFrame for easier analysis and parse the nested output JSON to extract detailed metadata fields.

In [ ]:
# Build DataFrame from examples
df = pd.DataFrame(examples)

# Parse the nested output JSON to get detailed fields
output_records = [json.loads(e["output"]) for e in examples]
df_detail = pd.DataFrame(output_records)

# Show a summary table of key columns
summary_cols = [
    "treebank_id", "language_name", "wals_word_order", "glottolog_family_name",
    "macroarea", "modality", "case_richness_count", "head_direction_entropy_binary",
    "num_tokens"
]
print("Sample treebanks:")
print(df_detail[summary_cols].head(10).to_string(index=False))

## Visualizations

Plot distributions of word orders, language families, macroareas, and the relationship between case richness and head-direction entropy.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Word order distribution
wo_col = df_detail["wals_word_order"].fillna("Unknown")
wo_counts = wo_col.value_counts()
colors_wo = plt.cm.Set2(np.linspace(0, 1, len(wo_counts)))
axes[0, 0].barh(wo_counts.index, wo_counts.values, color=colors_wo)
axes[0, 0].set_xlabel("Count")
axes[0, 0].set_title("WALS Word Order (Feature 81A)")
for i, v in enumerate(wo_counts.values):
    axes[0, 0].text(v + 0.2, i, str(v), va="center", fontsize=9)

# 2. Top language families
fam_col = df_detail["glottolog_family_name"].fillna("Unknown")
fam_counts = fam_col.value_counts().head(10)
colors_fam = plt.cm.tab10(np.linspace(0, 1, len(fam_counts)))
axes[0, 1].barh(fam_counts.index, fam_counts.values, color=colors_fam)
axes[0, 1].set_xlabel("Count")
axes[0, 1].set_title("Top Language Families (Glottolog)")
for i, v in enumerate(fam_counts.values):
    axes[0, 1].text(v + 0.1, i, str(v), va="center", fontsize=9)

# 3. Macroarea distribution
ma_col = df_detail["macroarea"].fillna("Unknown")
ma_counts = ma_col.value_counts()
colors_ma = plt.cm.Pastel1(np.linspace(0, 1, len(ma_counts)))
axes[1, 0].bar(range(len(ma_counts)), ma_counts.values, color=colors_ma)
axes[1, 0].set_xticks(range(len(ma_counts)))
axes[1, 0].set_xticklabels(ma_counts.index, rotation=35, ha="right", fontsize=8)
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Macroarea Distribution")

# 4. Case richness vs. head-direction entropy (colored by word order)
wo_labels = df_detail["wals_word_order"].fillna("Unknown")
unique_wo = sorted(wo_labels.unique(), key=lambda x: (x == "Unknown", x))
cmap = plt.cm.Set1(np.linspace(0, 0.8, len(unique_wo)))
wo_color_map = {wo: cmap[i] for i, wo in enumerate(unique_wo)}
for wo in unique_wo:
    mask = wo_labels == wo
    axes[1, 1].scatter(
        df_detail.loc[mask, "case_richness_count"],
        df_detail.loc[mask, "head_direction_entropy_binary"],
        c=[wo_color_map[wo]], label=wo, alpha=0.7, edgecolors="k", linewidths=0.5, s=50
    )
axes[1, 1].set_xlabel("Case Richness (distinct values)")
axes[1, 1].set_ylabel("Head-Direction Entropy (binary)")
axes[1, 1].set_title("Case Richness vs. Head-Direction Entropy")
axes[1, 1].legend(title="Word Order", fontsize=7, title_fontsize=8, loc="lower left")

plt.tight_layout()
plt.savefig("typological_overview.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved typological_overview.png")